# Auditoría: construcción del corpus ampliado (3.259 filas)

Este notebook reúne, en orden de ejecución, **todo el código** que se usó para
llevar `dataset_politica_colombiana.xlsx` de 793 filas (el corpus original
heredado del curso) a 3.259 filas balanceadas (1.633 `TRUE` / 1.626 `FALSE`).

No es un notebook de entrenamiento — no ajusta ningún modelo. Es un registro
de auditoría: cada celda es el código real que se ejecutó, reconstruido a
partir del historial de `git` de este repositorio (commit `db1771f`) y de la
sesión de trabajo que lo generó, para que cualquiera pueda revisar exactamente
de dónde salió cada fila nueva del dataset y por qué.

**Orden del procedimiento:**
1. Diagnóstico del corpus original (¿está bien etiquetado lo que ya había?)
2. Cobertura de sitios satíricos → 638 filas nuevas (`FALSE`)
3. Correcciones de calidad sobre esas 638 filas
4. Verificador de hechos Colombiacheck → 599 filas nuevas (`FALSE`)
5. Noticias reales de El Espectador para balancear → 1.230 filas nuevas (`TRUE`)
6. Verificación final del corpus

Ver `README.md` para el contexto completo, incluyendo qué fuentes se
evaluaron y se **descartaron explícitamente** (La Silla Vacía, AFP Factual,
El Tiempo, Portafolio) y por qué.

> Requiere `openpyxl`, `beautifulsoup4` (`pip install openpyxl beautifulsoup4`).
> Las celdas de scraping hacen peticiones HTTP reales a los sitios de origen;
> pueden tardar varios minutos y su resultado exacto puede variar si se
> vuelven a ejecutar en otra fecha (los sitios siguen publicando contenido).


## 1. Diagnóstico del corpus original

Antes de tocar nada: ¿las filas del corpus original que ya venían de sitios
satíricos conocidos (`actualidadpanamericana.com`, `elchiguirebipolar.net`)
estaban bien etiquetadas como `FALSE`? (Resultado: sí, el 100 %.)


In [ ]:
"""
Check whether every article sourced from known Colombian satirical/parody
news sites in dataset_politica_colombiana.xlsx is labeled as fake (label=False).

Known satirical domains present in this corpus (identified by manual review
of the URL column): actualidadpanamericana.com and elchiguirebipolar.net.
Both are self-described satire outlets, not real news.
"""
import os
from urllib.parse import urlparse
from openpyxl import load_workbook

PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "dataset_politica_colombiana.xlsx")

SATIRICAL_DOMAINS = {
    "actualidadpanamericana.com",
    "elchiguirebipolar.net",
}


def domain_of(url: str) -> str:
    if not url:
        return ""
    netloc = urlparse(url).netloc.lower()
    return netloc[4:] if netloc.startswith("www.") else netloc


def normalize_label(raw):
    """Handle the label column being stored as text 'TRUE'/'FALSE' rather
    than a numeric 0/1, which is what actually happens in this xlsx."""
    if isinstance(raw, bool):
        return raw
    if isinstance(raw, (int, float)):
        return bool(raw)
    if isinstance(raw, str):
        s = raw.strip().upper()
        if s in ("TRUE", "1", "REAL"):
            return True
        if s in ("FALSE", "0", "FALSA", "FAKE"):
            return False
    return None  # unrecognized


def main():
    wb = load_workbook(PATH, read_only=True, data_only=True)
    ws = wb.active

    rows = list(ws.iter_rows(values_only=True))
    header = [str(h).strip().lower() if h is not None else "" for h in rows[0]]
    data = rows[1:]

    col = {name: idx for idx, name in enumerate(header)}
    required = {"id", "label", "title", "url"}
    missing = required - set(col)
    if missing:
        raise SystemExit(f"Missing expected columns: {missing}. Found: {header}")

    total = 0
    satirical_rows = []
    label_parse_failures = []

    for r in data:
        if r is None or all(v is None for v in r):
            continue
        total += 1
        url = r[col["url"]] if col.get("url") is not None else None
        dom = domain_of(str(url)) if url else ""
        if dom in SATIRICAL_DOMAINS:
            raw_label = r[col["label"]]
            label = normalize_label(raw_label)
            if label is None:
                label_parse_failures.append((r[col["id"]], raw_label))
            satirical_rows.append({
                "id": r[col["id"]],
                "domain": dom,
                "title": r[col["title"]],
                "raw_label": raw_label,
                "label_is_real": label,
            })

    print(f"Total rows read: {total}")
    print(f"Rows sourced from known satirical domains: {len(satirical_rows)}")
    print(f"Domains checked: {sorted(SATIRICAL_DOMAINS)}\n")

    if label_parse_failures:
        print("!! Could not parse label for these rows:")
        for _id, raw in label_parse_failures:
            print(f"   id={_id} raw_label={raw!r}")
        print()

    mislabeled = [row for row in satirical_rows if row["label_is_real"] is True]
    correctly_labeled = [row for row in satirical_rows if row["label_is_real"] is False]

    print(f"Correctly labeled as FAKE (label=False/0): {len(correctly_labeled)}")
    print(f"INCORRECTLY labeled as REAL (label=True/1): {len(mislabeled)}\n")

    if mislabeled:
        print("Satirical-source rows labeled as REAL news (should be FAKE):")
        for row in mislabeled:
            print(f"  id={row['id']:>4}  domain={row['domain']:<30}  raw_label={row['raw_label']!r}")
            print(f"        title: {row['title']}")
    else:
        print("All rows sourced from satirical domains are correctly labeled as fake.")

    print("\nPer-domain breakdown:")
    for dom in sorted(SATIRICAL_DOMAINS):
        rows_d = [r for r in satirical_rows if r["domain"] == dom]
        fake_n = sum(1 for r in rows_d if r["label_is_real"] is False)
        real_n = sum(1 for r in rows_d if r["label_is_real"] is True)
        unk_n = sum(1 for r in rows_d if r["label_is_real"] is None)
        print(f"  {dom:<30} total={len(rows_d):<4} labeled_fake={fake_n:<4} labeled_real={real_n:<4} unparseable={unk_n}")


if __name__ == "__main__":
    main()


## 2. Cobertura de sitios satíricos y clasificación política

Recorre los sitemaps XML públicos de los dos sitios satíricos para ver qué
tanto de su contenido político colombiano *no* estaba ya en el corpus, y
clasifica cada artículo candidato como sátira política colombiana o no,
usando el título, la descripción y el cuerpo real del artículo (no solo el
slug de la URL). Este script pasó por varias iteraciones para eliminar
falsos positivos (palabras genéricas como "gobierno" o "policía" usadas
como chiste en cualquier tema) — la versión de abajo es la final.

Salida: `satirical_candidates_political.csv` (638 filas tras las
correcciones), `satirical_candidates_other.csv` (sátira no política,
descartada), `satirical_fetch_failures.csv`.


In [ ]:
"""
Cross-check dataset_politica_colombiana.xlsx against the live sitemaps of the
two known satirical sources already present in the corpus
(actualidadpanamericana.com, elchiguirebipolar.net) to find Colombia-political
satire that never made it into the dataset.

Two-stage classification:
  Stage 1 (recall, cheap): scan every sitemap URL's slug for any Colombia
      place/figure/institution token. This is deliberately broad — it's just
      there to shrink ~14,000 site-wide URLs down to a candidate set worth
      fetching, not the final answer.
  Stage 2 (precision, expensive): actually fetch each Stage-1 candidate page
      and read its real <title>, meta description, AND the first few
      paragraphs of the article body. Classify it as "Colombian political
      satire" if that text contains a Colombia signal AND a political-context
      term (government, elections, conflict, institutions, named politicians,
      municipal/administrative bodies, etc.) — or a signal that is itself
      unambiguously political (e.g. "farc", "congreso", a named president).
      All matching is whole-word / phrase, accent-insensitive, done on real
      article text — not URL substrings — specifically to avoid false hits
      like "petrolero" matching "petro" or "california" matching "cali".

      Earlier versions of this script classified off the meta description
      alone, which WordPress/Yoast truncates to ~155 characters — political
      signals that showed up later in the description or only in the body
      (e.g. "elecciones para la Alcaldía de Bogotá... destitución de Gustavo
      Petro" arriving after the truncation point) were missed. Pulling the
      first several body paragraphs fixes that class of false negative.

Output:
  - satirical_candidates_political.csv       -> classified as Colombian political satire, not in dataset
  - satirical_candidates_other.csv           -> Colombia-touching but NOT classified as political (sports, entertainment, etc.)
  - satirical_fetch_failures.csv             -> candidate URLs that couldn't be fetched/parsed
Console prints a summary. This is a classifier over public sitemap + article
metadata (title/meta description), not a guarantee — always skim before
merging any of this into the real dataset.
"""
import csv
import os
import re
import time
import unicodedata
import urllib.request
import urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse

from bs4 import BeautifulSoup
from openpyxl import load_workbook

REPO_DIR = os.path.dirname(os.path.abspath(__file__))
DATASET_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.xlsx")
OUT_POLITICAL = os.path.join(REPO_DIR, "satirical_candidates_political.csv")
OUT_OTHER = os.path.join(REPO_DIR, "satirical_candidates_other.csv")
OUT_FAILURES = os.path.join(REPO_DIR, "satirical_fetch_failures.csv")

UA = "Mozilla/5.0 (compatible; dataset-coverage-check/1.0; academic project)"
SITEMAP_REQUEST_DELAY = 0.5   # seconds between sitemap-index fetches (few, sequential)
ARTICLE_FETCH_WORKERS = 8     # concurrent article fetches (politeness vs. speed)
ARTICLE_TIMEOUT = 12

SITES = {
    "actualidadpanamericana.com": {
        "sitemap_index": "https://actualidadpanamericana.com/sitemap.xml",
    },
    "elchiguirebipolar.net": {
        "sitemap_index": "https://www.elchiguirebipolar.net/wp-sitemap.xml",
    },
}

# ---------------------------------------------------------------------------
# Stage 1: cheap slug-level recall filter (place/figure/institution tokens)
# ---------------------------------------------------------------------------
SLUG_TOKENS = {
    "colombia", "colombiano", "colombiana", "colombianos", "colombianas",
    "bogota", "bogotano", "bogotana", "medellin", "cali", "barranquilla",
    "cartagena", "bucaramanga", "cundinamarca", "antioquia", "atlantico",
    "santander", "boyaca", "narino", "cauca", "cordoba", "tolima", "huila",
    "petro", "duque", "uribe", "uribista", "uribismo", "santos", "samper",
    "pastrana", "gaitan", "farc", "eln", "dane", "dnp", "minhacienda",
    "senado", "registraduria", "contraloria", "procuraduria",
    "defensoria", "esmad", "transmilenio", "petrista", "petrismo",
    "gustavo-petro", "ivan-duque", "alvaro-uribe", "francia-marquez",
    "claudia-lopez", "rodolfo-hernandez", "sergio-fajardo",
    "federico-gutierrez", "juan-manuel-santos",
}

# ---------------------------------------------------------------------------
# Stage 2: content-level classification (checked against real title+description)
# ---------------------------------------------------------------------------

# Signals that are ALREADY political on their own (named politicians,
# political institutions — national AND municipal/administrative — armed
# political actors) — one hit is enough. Institution words are included
# bare (not just as "X de Bogotá" exact phrases) because satire headlines
# name plenty of cities/departments beyond Bogotá.
STRONG_POLITICAL_SIGNALS = {
    "petro", "gustavo petro", "duque", "ivan duque", "uribe", "alvaro uribe",
    "uribista", "uribismo", "petrista", "petrismo", "santos",
    "juan manuel santos", "samper", "pastrana", "francia marquez",
    "claudia lopez", "rodolfo hernandez", "sergio fajardo",
    "federico gutierrez", "farc", "eln", "esmad", "congreso", "senado",
    "camara de representantes", "concejo", "concejo de bogota",
    "procuraduria", "contraloria", "defensoria del pueblo",
    "registraduria", "corte constitucional", "corte suprema",
    "casa de narino", "corte penal", "gobernacion", "alcaldia",
    "alcaldia de bogota", "minhacienda", "ministerio de hacienda",
    "ministerio de defensa", "dnp", "distrito", "distrito capital",
    "personeria", "veeduria", "curul", "concejal", "diputado",
    "paro nacional", "paro armado", "paro civico", "movilizacion social",
    "consulta popular", "consejo de estado", "junta directiva",
}

# Weak Colombia signals (place names) — need a political-context term too,
# since "Bogotá" or "Medellín" on their own could be sports/culture/crime satire.
WEAK_COLOMBIA_SIGNALS = {
    "colombia", "colombiano", "colombiana", "colombianos", "colombianas",
    "bogota", "bogotano", "bogotana", "medellin", "cali", "barranquilla",
    "cartagena", "bucaramanga", "cundinamarca", "antioquia",
    "valle del cauca", "atlantico", "santander", "boyaca", "narino",
    "cauca", "cordoba", "tolima", "huila", "dane",
}

POLITICAL_CONTEXT_TERMS = {
    # NOTE: bare "gobierno", "policia", "politica"/"politico" were tried and
    # dropped — this site uses them as a stock punchline ("el gobierno no
    # autorizo...") in jokes about anything (airline routes, perfume
    # launches), so on their own they're not a reliable political signal.
    # Specific phrases keep the genuine government-story recall without that
    # noise.
    "gobierno colombiano", "gobierno nacional", "gobierno distrital",
    "gobierno de bogota", "policia nacional", "presidente", "presidencia",
    "ministro", "ministra", "ministerio", "congreso", "senado", "senador",
    "senadora", "camara de representantes", "representante a la camara",
    "alcaldia", "alcalde", "alcaldesa", "gobernador", "gobernadora",
    "gobernacion", "corte constitucional", "corte suprema", "procuraduria",
    "contraloria", "defensoria", "registraduria", "eleccion", "elecciones",
    "campana", "campaña", "candidato", "candidata", "reforma tributaria",
    "reforma pensional", "reforma laboral", "reforma a la salud", "reforma",
    "decreto", "proyecto de ley", "impuesto", "impuestos", "paz total",
    "acuerdo de paz", "guerrilla", "paramilitar", "ejercito",
    "partido politico", "oposicion", "coalicion", "bancada",
    "concejal", "diputado", "mocion de censura", "revocatoria",
    "plebiscito", "referendo", "consulta popular", "corrupcion",
    "escandalo politico",
    "distrito", "secretaria de educacion", "secretaria de movilidad",
    "secretaria de salud", "secretaria de gobierno", "pot",
    "ordenamiento territorial", "licitacion", "contratacion publica",
    "obra publica", "paro nacional", "paro civico", "paro de transporte",
    "marcha de protesta", "protesta social", "movilizacion social",
    "curul", "acto legislativo",
    "salario minimo", "iva", "personeria", "veeduria",
}


def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def normalize_text(s: str) -> str:
    return strip_accents(s.lower())


def has_any_phrase(text: str, phrases) -> bool:
    """Whole-word/phrase match on normalized text (word-boundary regex),
    so 'petro' doesn't match inside 'petrolero' and 'cali' doesn't match
    inside 'california'."""
    for phrase in phrases:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        if re.search(pattern, text):
            return True
    return False


def fetch_url(url: str, timeout: int = ARTICLE_TIMEOUT, retries: int = 3) -> str:
    """Fetch with retry/backoff — actualidadpanamericana.com has shown
    transient timeouts under sustained request volume in this session,
    most likely mild rate-limiting rather than a real outage."""
    last_exc = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception as e:
            last_exc = e
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
    raise last_exc


def extract_locs(xml_text: str):
    return re.findall(r"<loc>([^<]+)</loc>", xml_text)


def collect_all_urls(sitemap_index_url: str):
    index_xml = fetch_url(sitemap_index_url, timeout=20)
    time.sleep(SITEMAP_REQUEST_DELAY)
    sub_sitemaps = [u for u in extract_locs(index_xml) if "image-sitemap" not in u]

    all_urls = []
    for sm_url in sub_sitemaps:
        try:
            xml_text = fetch_url(sm_url, timeout=20)
        except Exception as e:
            print(f"    !! failed to fetch {sm_url}: {e}")
            continue
        time.sleep(SITEMAP_REQUEST_DELAY)
        locs = extract_locs(xml_text)
        if "<sitemapindex" in xml_text:
            for nested in locs:
                try:
                    nested_xml = fetch_url(nested, timeout=20)
                    all_urls.extend(extract_locs(nested_xml))
                except Exception as e:
                    print(f"    !! failed to fetch nested {nested}: {e}")
                time.sleep(SITEMAP_REQUEST_DELAY)
        else:
            all_urls.extend(locs)
    return all_urls


def normalize_url(url: str) -> str:
    p = urlparse(url.strip())
    netloc = p.netloc.lower()
    if netloc.startswith("www."):
        netloc = netloc[4:]
    path = p.path.rstrip("/")
    return f"{netloc}{path}"


def domain_of(url: str) -> str:
    netloc = urlparse(url).netloc.lower()
    return netloc[4:] if netloc.startswith("www.") else netloc


def slug_is_candidate(url: str) -> bool:
    slug = normalize_text(url)
    tokens = re.split(r"[^a-z0-9]+", slug)
    return any(tok in SLUG_TOKENS for tok in tokens)


def load_dataset_urls():
    wb = load_workbook(DATASET_PATH, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    header = [str(h).strip().lower() if h is not None else "" for h in rows[0]]
    col = {name: idx for idx, name in enumerate(header)}
    if "url" not in col:
        raise SystemExit(f"No 'url' column found. Header: {header}")

    urls_by_domain = {}
    for r in rows[1:]:
        if r is None:
            continue
        url = r[col["url"]]
        if not url:
            continue
        url = str(url)
        dom = domain_of(url)
        urls_by_domain.setdefault(dom, set()).add(normalize_url(url))
    return urls_by_domain


BODY_PARAGRAPHS_TO_READ = 6  # enough to cover the lede on these sites without pulling whole pages

def fetch_article_content(url: str):
    """Returns (title, description, body_excerpt). `description` is the
    (often truncated) meta description; `body_excerpt` is the first few
    real paragraphs of the article, which is what actually catches signals
    that show up after the meta description's ~155-character cutoff."""
    html = fetch_url(url)
    soup = BeautifulSoup(html, "html.parser")

    title = ""
    if soup.title and soup.title.string:
        title = soup.title.string.strip()
    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        title = og_title["content"].strip() or title

    description = ""
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        description = meta_desc["content"].strip()
    og_desc = soup.find("meta", property="og:description")
    if og_desc and og_desc.get("content"):
        description = og_desc["content"].strip() or description

    paras = soup.select("div.entry-content p") or soup.select("article p")
    body_excerpt = " ".join(
        p.get_text(" ", strip=True) for p in paras[:BODY_PARAGRAPHS_TO_READ]
    )

    return title, description, body_excerpt


def classify(title: str, description: str, body_excerpt: str = ""):
    """Returns (is_political: bool, reason: str)."""
    text = normalize_text(f"{title} . {description} . {body_excerpt}")

    if has_any_phrase(text, STRONG_POLITICAL_SIGNALS):
        return True, "strong_political_signal"

    has_place = has_any_phrase(text, WEAK_COLOMBIA_SIGNALS)
    has_context = has_any_phrase(text, POLITICAL_CONTEXT_TERMS)
    if has_place and has_context:
        return True, "place+political_context"

    return False, "no_political_signal" if not has_place else "place_only_no_political_context"


def fetch_and_classify(url: str):
    try:
        title, description, body_excerpt = fetch_article_content(url)
    except Exception as e:
        return {"url": url, "error": str(e)}
    is_political, reason = classify(title, description, body_excerpt)
    return {
        "url": url,
        "title": title,
        "description": description,
        "is_political": is_political,
        "reason": reason,
    }


def main():
    print("Loading existing dataset URLs...")
    dataset_urls = load_dataset_urls()
    for dom, urls in dataset_urls.items():
        if dom in SITES:
            print(f"  {dom}: {len(urls)} URLs already in dataset")
    print()

    # ---- Stage 1: build candidate set per domain (slug recall filter, minus what's already in) ----
    stage1_candidates = []  # (domain, url)
    for domain, info in SITES.items():
        print(f"Crawling sitemap for {domain} ...")
        try:
            all_urls = collect_all_urls(info["sitemap_index"])
        except Exception as e:
            print(f"  !! could not crawl {domain}: {e}")
            continue
        print(f"  Total URLs on site: {len(all_urls)}")

        existing = dataset_urls.get(domain, set())
        slug_matches = [u for u in all_urls if slug_is_candidate(u)]
        new_ones = [u for u in slug_matches if normalize_url(u) not in existing]
        print(f"  Slug-level Colombia recall filter: {len(slug_matches)} "
              f"({len(new_ones)} not already in dataset)\n")
        stage1_candidates.extend(new_ones)

    print(f"Stage 1 candidate set (to be content-checked): {len(stage1_candidates)}")
    print(f"Fetching each candidate's real title/description with "
          f"{ARTICLE_FETCH_WORKERS} concurrent workers...\n")

    # ---- Stage 2: fetch + classify by actual content ----
    political, other, failures = [], [], []
    done = 0
    with ThreadPoolExecutor(max_workers=ARTICLE_FETCH_WORKERS) as pool:
        futures = {pool.submit(fetch_and_classify, url): url for url in stage1_candidates}
        for fut in as_completed(futures):
            result = fut.result()
            done += 1
            if done % 100 == 0 or done == len(stage1_candidates):
                print(f"  ...{done}/{len(stage1_candidates)} fetched")
            if "error" in result:
                failures.append(result)
            elif result["is_political"]:
                political.append(result)
            else:
                other.append(result)

    domain_of_url = lambda u: domain_of(u)

    print("\n" + "=" * 70)
    print("RESULTS")
    print("=" * 70)
    print(f"Fetched successfully : {len(political) + len(other)}")
    print(f"Fetch failures        : {len(failures)}")
    print(f"Classified POLITICAL  : {len(political)}")
    print(f"Classified other      : {len(other)}")

    for dom in SITES:
        n_pol = sum(1 for r in political if domain_of_url(r["url"]) == dom)
        n_oth = sum(1 for r in other if domain_of_url(r["url"]) == dom)
        n_fail = sum(1 for r in failures if domain_of_url(r["url"]) == dom)
        print(f"\n  {dom}:")
        print(f"    political : {n_pol}")
        print(f"    other     : {n_oth}")
        print(f"    failed    : {n_fail}")

    def write_csv(path, rows, fields):
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            for r in rows:
                writer.writerow({k: r.get(k, "") for k in fields})

    write_csv(OUT_POLITICAL, political, ["url", "title", "description", "reason"])
    write_csv(OUT_OTHER, other, ["url", "title", "description", "reason"])
    write_csv(OUT_FAILURES, failures, ["url", "error"])

    print(f"\nWritten:")
    print(f"  {OUT_POLITICAL}  ({len(political)} rows)")
    print(f"  {OUT_OTHER}  ({len(other)} rows)")
    print(f"  {OUT_FAILURES}  ({len(failures)} rows)")

    if political:
        print("\nSample of classified-political candidates (first 10):")
        for r in political[:10]:
            print(f"  [{domain_of_url(r['url'])}] {r['title']}")
            print(f"      {r['url']}")


if __name__ == "__main__":
    main()


## 3. Fusión de la sátira política al dataset

Toma `satirical_candidates_political.csv`, reconstruye fecha de publicación
y descripción a partir del cuerpo real de cada artículo (la metadescripción
de WordPress viene truncada a ~155 caracteres y a veces cortaba la señal
política a la mitad), y agrega cada fila con `label=FALSE`.


In [ ]:
"""
Merge satirical_candidates_political.csv into dataset_politica_colombiana.xlsx.

For each candidate URL (already classified as Colombian political satire and
confirmed absent from the dataset by scrape_satirical_sources.py), this:
  1. Re-fetches the article to get a clean body-derived description (matches
     the dataset's own style better than a truncated meta description) and a
     publish date.
       - elchiguirebipolar.net embeds its publish date directly in the URL
         path (DD-MM-YYYY/slug), no fetch needed for that part.
       - actualidadpanamericana.com prints it in a "Publicado el D mes, YYYY"
         byline at the top of the article body.
  2. Cleans the title (strips the site's " - Actualidad Panamericana" /
     " | El Chigüire Bipolar" suffix).
  3. Appends a new row (id, label=FALSE, title, description, date, url) to
     the dataset, continuing the id sequence from the current max.

Before writing, the original file is copied to
dataset_politica_colombiana.backup_pre_expansion.xlsx so the merge can be
undone if something looks wrong after review.
"""
import csv
import os
import re
import shutil
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import openpyxl

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from scrape_satirical_sources import (  # noqa: E402
    fetch_url, domain_of, normalize_url, ARTICLE_FETCH_WORKERS,
)
from bs4 import BeautifulSoup  # noqa: E402

REPO_DIR = os.path.dirname(os.path.abspath(__file__))
DATASET_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.xlsx")
BACKUP_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.backup_pre_expansion.xlsx")
CANDIDATES_PATH = os.path.join(REPO_DIR, "satirical_candidates_political.csv")
FAILURES_PATH = os.path.join(REPO_DIR, "dataset_merge_failures.csv")

TITLE_SUFFIX_RE = re.compile(
    r"\s*[-|]\s*(Actualidad Panamericana|El Chig[üu]ire Bipolar)\s*$", re.IGNORECASE
)

MONTHS_ES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}
AP_DATE_RE = re.compile(
    r"Publicado el\s+(\d{1,2})\s*(?:de\s+)?([A-Za-zñÁÉÍÓÚáéíóú]+)\s*,?\s*(?:de\s+)?(\d{4})",
    re.IGNORECASE,
)
CHIGUIRE_URL_DATE_RE = re.compile(r"/(\d{2})-(\d{2})-(\d{4})/")


def clean_title(raw_title: str) -> str:
    return TITLE_SUFFIX_RE.sub("", raw_title or "").strip()


def extract_ap_content(url: str):
    """Returns (description, date_str) for an actualidadpanamericana.com article."""
    html = fetch_url(url)
    soup = BeautifulSoup(html, "html.parser")
    paras = soup.select("div.entry-content p") or soup.select("article p")
    full_text = " ".join(p.get_text(" ", strip=True) for p in paras)

    date_str = ""
    m = AP_DATE_RE.search(full_text)
    if m:
        day, month_name, year = m.groups()
        month_num = MONTHS_ES.get(month_name.lower())
        if month_num:
            date_str = f"{int(day):02d}/{month_num:02d}/{year}"

    # Strip the "Publicado el ... ." byline prefix to get the real lede.
    body = AP_DATE_RE.sub("", full_text, count=1)
    body = re.sub(r"^\s*por\s+\S+\s+en\s+[^.]*\.\s*", "", body, flags=re.IGNORECASE)
    body = body.strip()

    description = body[:500].strip()
    if description and not description.endswith((".", "…", "!", "?")):
        # cut at the last full sentence within the first 500 chars if possible
        last_period = description.rfind(". ")
        if last_period > 80:
            description = description[: last_period + 1]
    return description, date_str


def extract_chiguire_date(url: str) -> str:
    m = CHIGUIRE_URL_DATE_RE.search(url)
    if not m:
        return ""
    day, month, year = m.groups()
    return f"{day}/{month}/{year}"


def main():
    print("Loading candidates...")
    with open(CANDIDATES_PATH, newline="", encoding="utf-8") as f:
        candidates = list(csv.DictReader(f))
    print(f"  {len(candidates)} political candidates to merge")

    print("Loading dataset...")
    wb = openpyxl.load_workbook(DATASET_PATH)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    header = rows[0]
    existing_ids = [int(r[0]) for r in rows[1:] if r[0] is not None]
    existing_urls = {normalize_url(str(r[5])) for r in rows[1:] if r[5]}
    next_id = max(existing_ids) + 1
    print(f"  {len(existing_ids)} existing rows, next id = {next_id}")
    print(f"  header: {header}")

    print(f"\nBacking up original dataset to {BACKUP_PATH}")
    shutil.copyfile(DATASET_PATH, BACKUP_PATH)

    to_process = []
    skipped_dupe = 0
    for cand in candidates:
        if normalize_url(cand["url"]) in existing_urls:
            skipped_dupe += 1
        else:
            to_process.append(cand)

    def process_one(cand):
        url = cand["url"]
        dom = domain_of(url)
        title = clean_title(cand["title"])

        try:
            if dom == "elchiguirebipolar.net":
                date_str = extract_chiguire_date(url)
                description = cand.get("description", "").strip()
                if len(description) < 40:
                    html = fetch_url(url)
                    soup = BeautifulSoup(html, "html.parser")
                    paras = [p.get_text(" ", strip=True) for p in soup.select("article p")]
                    paras = [p for p in paras if len(p) > 40]
                    description = paras[0][:500] if paras else description
            elif dom == "actualidadpanamericana.com":
                description, date_str = extract_ap_content(url)
                if len(description) < 40:
                    description = cand.get("description", "").strip()
            else:
                return {"url": url, "error": f"unknown domain {dom}"}
        except Exception as e:
            return {"url": url, "error": str(e)}

        if not description:
            description = title  # last-resort fallback, never leave it empty

        return {"title": title, "description": description, "date": date_str, "url": url}

    print(f"\nFetching/processing {len(to_process)} candidates with "
          f"{ARTICLE_FETCH_WORKERS} concurrent workers...")

    results, failures = [], []
    done = 0
    with ThreadPoolExecutor(max_workers=ARTICLE_FETCH_WORKERS) as pool:
        futures = {pool.submit(process_one, c): c for c in to_process}
        for fut in as_completed(futures):
            r = fut.result()
            done += 1
            if "error" in r:
                failures.append(r)
            else:
                results.append(r)
            if done % 100 == 0 or done == len(to_process):
                print(f"  ...{done}/{len(to_process)} processed "
                      f"({len(results)} ok, {len(failures)} failed)")

    # Assign ids in a stable order (by original candidate order) now that all
    # fetches are done — concurrency only affected fetch order, not this.
    url_order = {c["url"]: i for i, c in enumerate(to_process)}
    results.sort(key=lambda r: url_order.get(r["url"], 0))

    new_rows = []
    for r in results:
        new_rows.append({
            "id": str(next_id),
            "label": "FALSE",
            "title": r["title"],
            "description": r["description"],
            "date": r["date"],
            "url": r["url"],
        })
        next_id += 1

    print(f"\nAppending {len(new_rows)} new rows to the dataset...")
    for r in new_rows:
        ws.append([r["id"], r["label"], r["title"], r["description"], r["date"], r["url"]])

    wb.save(DATASET_PATH)
    print(f"Saved: {DATASET_PATH}")
    print(f"New row count: {len(existing_ids) + len(new_rows)} "
          f"({len(existing_ids)} original + {len(new_rows)} added)")

    if failures:
        with open(FAILURES_PATH, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["url", "error"])
            writer.writeheader()
            writer.writerows(failures)
        print(f"\n{len(failures)} candidates failed to merge — see {FAILURES_PATH}")

    if skipped_dupe:
        print(f"{skipped_dupe} candidates were already in the dataset (skipped as duplicates)")

    print("\nSample of newly added rows:")
    for r in new_rows[:5]:
        print(f"  id={r['id']}  date={r['date'] or '(none)'}  {r['title']}")
        print(f"    {r['description'][:150]}")


if __name__ == "__main__":
    main()


## 3b. Correcciones de calidad post-fusión

Dos problemas se detectaron **después** de fusionar las 638 filas de sátira,
revisando una muestra del resultado, y se corrigieron directamente sobre
`dataset_politica_colombiana.xlsx`:

1. **Encabezado de autor filtrado en la descripción.** El regex que quitaba
   la línea "Publicado el ... por &lt;autor&gt; en &lt;categoría&gt;." asumía
   un nombre de autor de una sola palabra; con nombres de dos palabras
   (p. ej. "Aquiles Baeza") el regex fallaba y la descripción quedaba con
   ese encabezado pegado al principio. Afectó 66 de las 638 filas nuevas.
2. **Una fila no era un artículo.** La URL
   `elchiguirebipolar.net/etiqueta/juan-manuel-santos/` es una página de
   archivo por etiqueta de WordPress (lista todos los artículos con esa
   etiqueta), no una noticia individual — pasó el filtro porque el slug
   contenía el nombre de un político. Se identificó y se eliminó esa fila.


In [ ]:
import openpyxl, re

wb = openpyxl.load_workbook("dataset_politica_colombiana.xlsx")
ws = wb.active

# --- 1. Limpiar encabezados de autor filtrados en la descripcion ---
BYLINE_RE = re.compile(r"^\s*por\s+.+?\s+en\s+.+?\.\s*", re.IGNORECASE)

fixed = 0
still_bad = []
for row in ws.iter_rows(min_row=2):
    id_cell, label_cell, title_cell, desc_cell, date_cell, url_cell = row
    if id_cell.value is None or int(id_cell.value) < 794:
        continue  # solo las filas agregadas en el paso 3 (ids >= 794)
    desc = desc_cell.value or ""
    if re.match(r"^\s*por\s+", desc, re.IGNORECASE):
        new_desc = BYLINE_RE.sub("", desc, count=1).strip()
        if new_desc and not re.match(r"^\s*por\s+", new_desc, re.IGNORECASE):
            desc_cell.value = new_desc
            fixed += 1
        else:
            still_bad.append((id_cell.value, desc))

print(f"Corregidas {fixed} filas por regex automatico")
print(f"Quedaron {len(still_bad)} sin corregir automaticamente: {[i for i,_ in still_bad]}")
# De las que quedaron, 2 eran falsos positivos (oraciones reales que
# empiezan con "Por..." como "Por orden de...", no un encabezado de autor).
# Solo id=1220 tenia un encabezado real sin limpiar; se corrigio a mano:
for row in ws.iter_rows(min_row=2):
    id_cell, label_cell, title_cell, desc_cell, date_cell, url_cell = row
    if id_cell.value == "1220":
        text = desc_cell.value
        marker = "Sociales . "
        idx = text.find(marker)
        if idx != -1:
            desc_cell.value = text[idx + len(marker):].strip()

wb.save("dataset_politica_colombiana.xlsx")
print("Guardado.")


In [ ]:
import openpyxl

wb = openpyxl.load_workbook("dataset_politica_colombiana.xlsx")
ws = wb.active

# --- 2. Eliminar la fila que era una pagina de archivo, no un articulo ---
target_row_idx = None
for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
    if row[0].value == "1430":
        target_row_idx = i
        break

if target_row_idx:
    print("Eliminando fila:", [c.value for c in ws[target_row_idx]])
    ws.delete_rows(target_row_idx)
    wb.save("dataset_politica_colombiana.xlsx")
    print("Guardado.")
else:
    print("Fila id=1430 no encontrada (ya fue eliminada).")


## 4. Verificador de hechos: Colombiacheck

Colombiacheck publica cada verificación con metadatos estructurados
`schema.org/ClaimReview` (afirmación revisada, calificación, fecha) — mucho
más confiable que raspar texto visible. Se excluyen las calificadas como
verdaderas; todo lo demás (Falso, Cuestionable, etc.) sigue la misma
convención que ya traía el corpus original: cualquier afirmación revisada
por un verificador de hechos se etiqueta `FALSE`.

**Nota de alcance:** de los tres verificadores de hechos representados en
el corpus original, solo se scrapeó este. La Silla Vacía se descartó porque
su `robots.txt` bloquea explícitamente a `ClaudeBot`/`anthropic-ai` por
nombre; AFP Factual se descartó porque incluso una petición simple de
`robots.txt` devuelve `403` desde su protección de bots. Ver README.md.

Salida: `colombiacheck_candidates.csv` (599 filas).


In [ ]:
"""
Find Colombiacheck.com fact-checks (Colombian political claims rated false/
misleading) that aren't yet in dataset_politica_colombiana.xlsx.

Scope note: this script deliberately only targets colombiacheck.com. Two
other fact-checkers already represented in the dataset were evaluated and
excluded:
  - lasillavacia.com: its robots.txt explicitly disallows AI-training
    crawlers by name (ClaudeBot, anthropic-ai, GPTBot, CCBot, etc.) under a
    section titled "Entrenamiento de IA (no Google) - BLOQUEADOS". Since this
    script builds ML training data, that's exactly the use their policy
    targets, so it is not scraped here.
  - factual.afp.com: returns HTTP 403 from Akamai bot-protection on even a
    plain robots.txt request. Not attempted.
  colombiacheck.com's robots.txt has no AI-specific restriction, just
  standard Drupal admin paths blocked.

Method:
  1. Colombiacheck embeds each fact-check's verdict as schema.org ClaimReview
     JSON-LD (claimReviewed, reviewBody, datePublished, reviewRating with an
     alternateName like "Falso"/"Cuestionable"/"Verdadero"). This is far more
     reliable than scraping visible text.
  2. Crawl the paginated /chequeos listing (9 articles/page) to collect
     article URLs.
  3. For each article, extract every ClaimReview block. Skip ones rated
     clearly true ("Verdadero"/"Cierto") — those aren't fake news. Everything
     else (Falso, Cuestionable, Engañoso, etc.) matches the dataset's own
     existing convention: every one of the 111 pre-existing Colombiacheck
     rows in the dataset is labeled FALSE regardless of the exact nuance of
     the original rating.
  4. Skip claims already present in the dataset (matched on URL + claim text).

Output: colombiacheck_candidates.csv — NOT auto-merged into the dataset.
Review before merging, same as the satire pipeline.
"""
import csv
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import openpyxl

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from scrape_satirical_sources import (  # noqa: E402
    fetch_url, normalize_text, has_any_phrase,
    STRONG_POLITICAL_SIGNALS, WEAK_COLOMBIA_SIGNALS, POLITICAL_CONTEXT_TERMS,
)

REPO_DIR = os.path.dirname(os.path.abspath(__file__))
DATASET_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.xlsx")
OUT_CSV = os.path.join(REPO_DIR, "colombiacheck_candidates.csv")

BASE = "https://colombiacheck.com"
LISTING_URL = BASE + "/chequeos"
MAX_LISTING_PAGES = 150          # ~1350 articles at 9/page
LISTING_WORKERS = 6
ARTICLE_WORKERS = 8

TRUE_RATINGS = {"verdadero", "cierto", "correcto"}

JSONLD_RE = re.compile(r'<script type="application/ld\+json">(.*?)</script>', re.S)


def get_listing_page_urls(page: int):
    html = fetch_url(f"{LISTING_URL}?page={page}", timeout=15)
    return sorted(set(re.findall(r'href="(/chequeos/[^"?#]+)"', html)))


def crawl_listing():
    print(f"Crawling up to {MAX_LISTING_PAGES} listing pages...")
    all_paths = set()
    empty_streak = 0
    with ThreadPoolExecutor(max_workers=LISTING_WORKERS) as pool:
        futures = {pool.submit(get_listing_page_urls, p): p for p in range(MAX_LISTING_PAGES)}
        results = {}
        for fut in as_completed(futures):
            p = futures[fut]
            try:
                results[p] = fut.result()
            except Exception as e:
                results[p] = []
                print(f"  page {p} failed: {e}")

    for p in range(MAX_LISTING_PAGES):
        paths = results.get(p, [])
        before = len(all_paths)
        all_paths.update(paths)
        if len(all_paths) == before:
            empty_streak += 1
        else:
            empty_streak = 0

    print(f"  collected {len(all_paths)} unique article paths")
    return [BASE + p for p in all_paths]


def extract_claims(url: str):
    """Returns list of dicts: claim, description, date_str, rating."""
    html = fetch_url(url, timeout=15)
    claims = []
    for block in JSONLD_RE.findall(html):
        try:
            data = json.loads(block)
        except Exception:
            continue
        graph = data.get("@graph", [])
        if isinstance(graph, dict):
            graph = [graph]
        if data.get("@type") == "ClaimReview":
            graph = graph + [data]
        for item in graph:
            if item.get("@type") != "ClaimReview":
                continue
            claim = (item.get("claimReviewed") or "").strip()
            if not claim:
                continue
            review_body = item.get("reviewBody") or ""
            if isinstance(review_body, list):
                review_body = " ".join(review_body)
            rating = item.get("reviewRating", {}) or {}
            alt_name = (rating.get("alternateName") or "").strip()
            date_raw = item.get("datePublished") or ""
            date_str = ""
            if date_raw:
                try:
                    date_str = datetime.fromisoformat(date_raw).strftime("%d/%m/%Y")
                except Exception:
                    pass
            claims.append({
                "claim": claim,
                "description": review_body.strip(),
                "date": date_str,
                "rating": alt_name,
                "url": url,
            })
    return claims


# The whole site is Colombia-scoped, so (unlike the satire-site classifier
# this borrows from) we don't also require a place-name signal here — that
# requirement existed to separate Colombia from the rest of Latin America on
# multi-country satire sites, which doesn't apply to a Colombia-only
# fact-checker. We do still filter for POLITICAL content specifically, since
# colombiacheck also fact-checks health/crime/viral-hoax claims unrelated to
# politics.
#
# Also extends the imported (older) political-figure list with current
# 2026-era names that postdate it, since Colombia had a presidential
# transition after the original list was built.
EXTRA_STRONG_SIGNALS = {
    "de la espriella", "abelardo de la espriella", "presidencia de la espriella",
}
EXTRA_CONTEXT_TERMS = {
    # NOTE: generic legal/crime words (condenado, sentencia, fiscalia, corte,
    # sancion) were tried and dropped here — same lesson as "gobierno"/
    # "policia" in scrape_satirical_sources.py: too common in ordinary,
    # non-political crime stories. "Contratista(s)" is kept because public
    # contracting fraud is specifically a political-corruption signal.
    "contratista", "contratistas", "contratacion publica", "presidenta",
}


def is_colombian_political(claim: str, description: str) -> bool:
    text = normalize_text(f"{claim} . {description}")
    if has_any_phrase(text, STRONG_POLITICAL_SIGNALS | EXTRA_STRONG_SIGNALS):
        return True
    return has_any_phrase(text, POLITICAL_CONTEXT_TERMS | EXTRA_CONTEXT_TERMS)


def load_existing_claims():
    wb = openpyxl.load_workbook(DATASET_PATH, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    header = [str(h).strip().lower() if h is not None else "" for h in rows[0]]
    col = {name: idx for idx, name in enumerate(header)}
    existing = set()
    for r in rows[1:]:
        if r is None:
            continue
        url = str(r[col["url"]] or "")
        title = normalize_text(str(r[col["title"]] or ""))
        if "colombiacheck.com" in url:
            existing.add((url.rstrip("/"), title))
    return existing


def main():
    print("Loading existing dataset claims (colombiacheck.com only)...")
    existing = load_existing_claims()
    print(f"  {len(existing)} existing (url, title) pairs from colombiacheck.com")

    urls = crawl_listing()

    print(f"\nFetching {len(urls)} articles with {ARTICLE_WORKERS} workers...")
    all_claims = []
    failures = 0
    done = 0
    with ThreadPoolExecutor(max_workers=ARTICLE_WORKERS) as pool:
        futures = {pool.submit(extract_claims, u): u for u in urls}
        for fut in as_completed(futures):
            done += 1
            try:
                all_claims.extend(fut.result())
            except Exception:
                failures += 1
            if done % 150 == 0 or done == len(urls):
                print(f"  ...{done}/{len(urls)} articles fetched "
                      f"({len(all_claims)} claims found so far, {failures} failed)")

    print(f"\nTotal claims extracted: {len(all_claims)}")

    true_rated = [c for c in all_claims if c["rating"].strip().lower() in TRUE_RATINGS]
    fake_rated = [c for c in all_claims if c["rating"].strip().lower() not in TRUE_RATINGS]
    print(f"  Rated true (excluded): {len(true_rated)}")
    print(f"  Rated false/misleading/other (candidate fake news): {len(fake_rated)}")

    political = [c for c in fake_rated if is_colombian_political(c["claim"], c["description"])]
    print(f"  Colombia-political among those: {len(political)}")

    new_ones = []
    for c in political:
        key = (c["url"].rstrip("/"), normalize_text(c["claim"]))
        if key not in existing:
            new_ones.append(c)
    print(f"  NOT already in dataset: {len(new_ones)}")

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["claim", "description", "date", "rating", "url"])
        writer.writeheader()
        writer.writerows(new_ones)

    print(f"\nWritten: {OUT_CSV} ({len(new_ones)} rows)")
    print("\nSample:")
    for c in new_ones[:10]:
        print(f"  [{c['rating']}] {c['claim'][:80]}")
        print(f"      {c['url']}")


if __name__ == "__main__":
    main()


## 5. Fusión de Colombiacheck al dataset

A diferencia de la sátira, aquí no hace falta reconstruir nada por
scraping adicional: `colombiacheck_candidates.csv` ya trae fecha y
descripción limpias extraídas directamente del `ClaimReview` estructurado.


In [ ]:
import csv, shutil, openpyxl

DATASET = "dataset_politica_colombiana.xlsx"
BACKUP = "dataset_politica_colombiana.backup_pre_colombiacheck_expansion.xlsx"
CANDIDATES = "colombiacheck_candidates.csv"

shutil.copyfile(DATASET, BACKUP)
print("Respaldo guardado en", BACKUP)

with open(CANDIDATES, newline="", encoding="utf-8") as f:
    candidates = list(csv.DictReader(f))
print(f"{len(candidates)} candidatos a fusionar")

wb = openpyxl.load_workbook(DATASET)
ws = wb.active
rows = list(ws.iter_rows(values_only=True))
existing_ids = [int(r[0]) for r in rows[1:] if r[0] is not None]
next_id = max(existing_ids) + 1
print(f"{len(existing_ids)} filas existentes, siguiente id = {next_id}")

added = 0
for c in candidates:
    ws.append([
        str(next_id),
        "FALSE",
        c["claim"].strip(),
        c["description"].strip(),
        c["date"],
        c["url"],
    ])
    next_id += 1
    added += 1

wb.save(DATASET)
print(f"Agregadas {added} filas. Total nuevo: {len(existing_ids) + added}")


## 6. Balanceo con noticias reales de El Espectador

Con el corpus en 1.626 filas `FALSE` y solo 403 `TRUE`, se agregan noticias
reales para cerrar la brecha. El Espectador se eligió porque ya era la
fuente real más representada en el corpus original (190 de 403), su
`robots.txt` no restringe bots de entrenamiento de IA (a diferencia de El
Tiempo y Portafolio, que sí bloquean `ClaudeBot`/`anthropic-ai`
explícitamente), y publica un sitemap dedicado a la sección "política" con
más de 10.000 artículos y fecha de publicación incluida en el XML.

**Nota técnica conservada a propósito:** la primera corrida de este script
tenía un error — las URLs de paginación del sitemap vienen con `&`
escapado como `&amp;` en el XML, y si no se des-escapan antes de usarlas
como URL de petición, el servidor ignora los parámetros de paginación
silenciosamente y devuelve la misma página siempre. Esa primera corrida
(defectuosa) agregó 100 filas; se corrigió el escapado y se corrió de nuevo
para las 1.130 filas restantes. La versión de abajo ya incluye la
corrección (`html.unescape`).


In [ ]:
"""
Pull real (TRUE-labeled) Colombian political news from elespectador.com to
balance the dataset's class distribution.

elespectador.com was chosen because:
  - It's already the dataset's single largest real-news source (190 of the
    original 403 TRUE rows).
  - Its robots.txt has no AI-training restriction of any kind (unlike
    eltiempo.com and portafolio.co, which explicitly block ClaudeBot/
    anthropic-ai by name and were excluded from this project for that
    reason).
  - It publishes a dedicated, well-structured sitemap for its "politica"
    section (Arc Publishing CMS), which means we get exact section
    membership for free — far more reliable than keyword-guessing whether an
    article is political, and each entry includes <lastmod> (publish date)
    directly in the sitemap, so no extra fetch is needed just for the date.

Method:
  1. Sample sitemap pages spread across the whole /politica/ archive (not
     just the most recent ones) for temporal diversity — same principle as
     the satire-site classifier, applied to a much cleaner signal (URL
     section) instead of keyword matching.
  2. For each URL not already in the dataset, fetch the article and pull its
     real <title>/og:title and meta description.
  3. Append as label=TRUE rows, continuing the dataset's id sequence.
"""
import csv
import html
import os
import re
import shutil
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import openpyxl
from bs4 import BeautifulSoup

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from scrape_satirical_sources import fetch_url, normalize_url  # noqa: E402

REPO_DIR = os.path.dirname(os.path.abspath(__file__))
DATASET_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.xlsx")
BACKUP_PATH = os.path.join(REPO_DIR, "dataset_politica_colombiana.backup_pre_realnews_expansion.xlsx")
OUT_CSV = os.path.join(REPO_DIR, "elespectador_real_news_added.csv")

SITEMAP_INDEX = "https://www.elespectador.com/arc/outboundfeeds/sitemap-index/section/politica/?outputType=xml"
ARTICLE_WORKERS = 8
TARGET_NEW_ROWS = 1130  # FALSE=1626, TRUE=503 after first (partial, buggy) run — this closes the gap

SITEMAP_ENTRY_RE = re.compile(
    r"<loc>([^<]+)</loc>\s*<lastmod>([^<]+)</lastmod>", re.S
)
TITLE_SUFFIX_RE = re.compile(r"\s*[|\-]\s*El Espectador\s*$", re.IGNORECASE)


def get_sitemap_page_entries(page_url: str):
    xml = fetch_url(page_url, timeout=20)
    return SITEMAP_ENTRY_RE.findall(xml)


def get_all_politica_subpages():
    index_xml = fetch_url(SITEMAP_INDEX, timeout=20)
    # sitemap XML HTML-escapes "&" as "&amp;" inside <loc> — these URLs get
    # used directly as request URLs, so they must be unescaped first or every
    # "&size=...&from=..." collapses into a malformed query string and the
    # server silently ignores the pagination, returning the same page
    # every time (this bit us on the first run: 21 "different" pages all
    # returned the same 100 URLs).
    return [html.unescape(u) for u in re.findall(r"<loc>([^<]+)</loc>", index_xml)]


def fetch_article(url: str):
    html = fetch_url(url)
    soup = BeautifulSoup(html, "html.parser")

    title = ""
    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        title = og_title["content"].strip()
    if not title and soup.title and soup.title.string:
        title = soup.title.string.strip()
    title = TITLE_SUFFIX_RE.sub("", title).strip()

    description = ""
    og_desc = soup.find("meta", property="og:description")
    if og_desc and og_desc.get("content"):
        description = og_desc["content"].strip()
    if not description:
        meta_desc = soup.find("meta", attrs={"name": "description"})
        if meta_desc and meta_desc.get("content"):
            description = meta_desc["content"].strip()

    return title, description


def load_existing_urls():
    wb = openpyxl.load_workbook(DATASET_PATH, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    existing_ids = [int(r[0]) for r in rows[1:] if r[0] is not None]
    existing_urls = {normalize_url(str(r[5])) for r in rows[1:] if r[5]}
    return existing_ids, existing_urls


def main():
    print("Loading existing dataset...")
    existing_ids, existing_urls = load_existing_urls()
    next_id = max(existing_ids) + 1
    print(f"  {len(existing_ids)} existing rows, next id = {next_id}")

    print("\nDiscovering politica sitemap sub-pages...")
    subpages = get_all_politica_subpages()
    print(f"  {len(subpages)} sub-pages available (100 URLs each, spans the full archive)")

    # Sample spread across the whole archive for temporal diversity, not just
    # the most recent batch.
    step = max(1, len(subpages) // 20)
    sampled_pages = subpages[::step]
    print(f"  sampling {len(sampled_pages)} pages spread across the archive")

    print("\nCollecting candidate URLs from sampled pages...")
    all_entries = {}
    for i, page_url in enumerate(sampled_pages, 1):
        try:
            entries = get_sitemap_page_entries(page_url)
        except Exception as e:
            print(f"  page {i} failed: {e}")
            continue
        for loc, lastmod in entries:
            if normalize_url(loc) not in existing_urls:
                all_entries[loc] = lastmod
        if i % 5 == 0:
            print(f"  ...{i}/{len(sampled_pages)} pages, {len(all_entries)} new candidate URLs so far")

    print(f"\nTotal new candidate URLs: {len(all_entries)}")
    candidates = list(all_entries.items())[:int(TARGET_NEW_ROWS * 1.15)]  # small buffer for failures
    print(f"Using {len(candidates)} candidates (buffered above target {TARGET_NEW_ROWS})")

    print(f"\nFetching {len(candidates)} articles with {ARTICLE_WORKERS} workers...")
    results = []
    failures = 0
    done = 0
    with ThreadPoolExecutor(max_workers=ARTICLE_WORKERS) as pool:
        futures = {pool.submit(fetch_article, url): (url, lastmod) for url, lastmod in candidates}
        for fut in as_completed(futures):
            url, lastmod = futures[fut]
            done += 1
            try:
                title, description = fut.result()
                if title and description and len(description) > 20:
                    try:
                        date_str = datetime.fromisoformat(lastmod.replace("Z", "+00:00")).strftime("%d/%m/%Y")
                    except Exception:
                        date_str = ""
                    results.append({"title": title, "description": description, "date": date_str, "url": url})
                else:
                    failures += 1
            except Exception:
                failures += 1
            if done % 150 == 0 or done == len(candidates):
                print(f"  ...{done}/{len(candidates)} fetched ({len(results)} ok, {failures} failed)")

    print(f"\nGot {len(results)} usable articles (target was {TARGET_NEW_ROWS})")
    results = results[:TARGET_NEW_ROWS]

    print(f"\nBacking up dataset to {BACKUP_PATH}")
    shutil.copyfile(DATASET_PATH, BACKUP_PATH)

    wb = openpyxl.load_workbook(DATASET_PATH)
    ws = wb.active
    for r in results:
        ws.append([str(next_id), "TRUE", r["title"], r["description"], r["date"], r["url"]])
        next_id += 1
    wb.save(DATASET_PATH)
    print(f"Saved. Added {len(results)} TRUE rows.")

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "description", "date", "url"])
        writer.writeheader()
        writer.writerows(results)
    print(f"Also wrote a record to {OUT_CSV}")

    print("\nSample:")
    for r in results[:5]:
        print(f"  [{r['date']}] {r['title'][:80]}")


if __name__ == "__main__":
    main()


## 7. Verificación final del corpus

Chequeo de sanidad sobre el resultado de todo el proceso.


In [ ]:
import openpyxl
from collections import Counter

wb = openpyxl.load_workbook("dataset_politica_colombiana.xlsx")
ws = wb.active
rows = list(ws.iter_rows(values_only=True))
header, data = rows[0], rows[1:]

print("Filas totales:", len(data))
print("Distribucion de label:", Counter(r[1] for r in data))

vacios_desc = sum(1 for r in data if not r[3] or len(str(r[3])) < 10)
vacios_fecha = sum(1 for r in data if not r[4])
vacios_titulo = sum(1 for r in data if not r[2])
print("Descripciones vacias/muy cortas:", vacios_desc)
print("Fechas vacias:", vacios_fecha)
print("Titulos vacios:", vacios_titulo)

urls = [r[5] for r in data]
print("URLs duplicadas en todo el corpus:", len(urls) - len(set(urls)))
